# 02 · Baseline captioning and decoding strategies (Milestone 2)

1. Train CLIP → GPT-2 with prefix conditioning (`configs/prefix_frozen.yaml`)
2. Read the training log and the captions written after every epoch
3. Compare greedy, beam search and nucleus sampling on 10 validation images

Train on a GPU (Colab or HPC). Check first with `--overfit-batch` and `--debug`.

In [ ]:
# Run from the repo root (works whether Jupyter starts in notebooks/ or the root).
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

# On Colab, run once instead:
# !git clone https://github.com/Yiting0210/clip-gpt2-captioning.git
# %cd clip-gpt2-captioning
# !pip install -q -r requirements.txt

import json, torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Train

Run these in a terminal, or uncomment them here. The first two take a minute and catch most bugs.

In [ ]:
# !python -m src.train --config configs/prefix_frozen.yaml --overfit-batch   # loss should approach 0
# !python -m src.train --config configs/prefix_frozen.yaml --debug           # whole pipeline, 3 steps
# !python -m src.train --config configs/prefix_frozen.yaml                   # full run
RUN = Path('runs/prefix_frozen')
events = [json.loads(l) for l in (RUN / 'train_log.jsonl').read_text(encoding='utf-8').splitlines()]
start = next(e for e in events if e['event'] == 'start')
{k: start[k] for k in ('device', 'injection', 'finetune', 'train_images', 'trainable', 'total', 'trainable_pct')}

## Training log

In [ ]:
import matplotlib.pyplot as plt
# Fixed categorical order (validated reference palette); recessive grid, no top/right spines.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=SERIES), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df",
                     "grid.linewidth": 0.8, "axes.edgecolor": "#9a9893", "axes.labelcolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e", "lines.linewidth": 2,
                     "figure.dpi": 110, "font.size": 10})

train = [e for e in events if e['event'] == 'train']
val = [e for e in events if e['event'] == 'val']
spe = start['steps_per_epoch']
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot([e['step'] / spe for e in train], [e['loss'] for e in train], color=SERIES[0], linewidth=1.2, alpha=0.7, label='train (batch)')
ax.plot([e['epoch'] for e in val], [e['val_loss'] for e in val], color=SERIES[1], marker='o', markersize=5, label='validation')
ax.set(xlabel='epoch', ylabel='cross-entropy loss', title='Training and validation loss')
ax.legend(frameon=False); plt.show()
[(e['epoch'], round(e['val_loss'], 3), e.get('epoch_time_s'), e.get('peak_mem_gb')) for e in val]

## Captions written after each epoch

The same validation images are captioned after every epoch, so you can watch the model learn.

In [ ]:
previews = [e for e in events if e['event'] == 'preview']
for iid in dict.fromkeys(p['image_id'] for p in previews):
    rows = [p for p in previews if p['image_id'] == iid]
    print(f'image {iid} | REF: {rows[0]["reference"]}')
    for p in rows:
        print(f'  epoch {p["epoch"]:>2}: {p["generated"]}')

## Decoding strategies

- **greedy**: highest-probability token at each step
- **beam_wK**: keep the K best partial captions; pick the best full one
- **nucleus_pP_tT**: sample from the smallest token set whose probability reaches P, after dividing logits by temperature T

In [ ]:
from IPython.display import Image as Show
from src import data
from src.decode import generate
from src.evaluate import save_gallery
from src.model import load_run

model, tok, cfg = load_run(RUN, DEVICE)
caps, splits = data.load_captions(), data.load_splits()
emb = data.load_embeddings(cfg['clip_model'])
ids10 = splits['val'][:10]
key = 'pooled' if cfg['injection'] == 'prefix' else 'patch'
feats = emb[key][[emb['index'][i] for i in ids10]].float().to(DEVICE)
SPECS = ['greedy', 'beam_w3', 'beam_w5', 'nucleus_p0.9_t0.7', 'nucleus_p0.9_t1.0']
out = {s: dict(zip(ids10, generate(model, feats, tok, s, seed=0))) for s in SPECS}
for i in ids10:
    print(f'{i} REF: {caps[i]["captions"][0]}')
    for s in SPECS:
        print(f'   {s:<18} {out[s][i]}')
Show(str(save_gallery(ids10[:6], out, caps, Path('results/m2_decoding_samples.png'))))

## Observations

*(Fill in after the run: does the loss plateau? which strategy reads best? where does the model fail?)*